In [ ]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper
from langchain_anthropic import ChatAnthropic
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import os
from datasets import Dataset

In [16]:
# Load environment files and store API keys
load_dotenv()
anthropic_key = os.getenv("ANTHROPIC_KEY")

In [27]:
# Read in dialogue data
file_path = "../../data/Hallucination/dialogue_data.json"
dialogue_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/general_data.json"
general_data = pd.read_json(file_path, lines=True)

file_path = "../../data/Hallucination/qa_data.json"
qa_data = pd.read_json(file_path, lines=True)

In [ ]:
# Generate a random choice (True for right_answer, False for hallucinated_answer)
choice = np.random.rand(len(qa_data)) < 0.5

# Assign the chosen answer
qa_data["selected_answer"] = np.where(choice, qa_data["right_answer"], qa_data["hallucinated_answer"])

# Add a column indicating the source of the answer
qa_data["hallucinated_flag"] = np.where(choice, 0, 1)

In [ ]:
# Set up Claude as the evaluator LLM
claude_llm = ChatAnthropic(api_key=anthropic_key, model="claude-3-7-sonnet-20250219")
evaluator_llm = LangchainLLMWrapper(claude_llm)

In [41]:
# Function to query the LLM with hallucination-aware prompt
async def evaluate_hallucination(context, question, response):
    
    from ragas.dataset_schema import SingleTurnSample 
    from ragas.metrics import Faithfulness

    sample = SingleTurnSample(
            user_input=question,
            response=response,
            retrieved_contexts=context
        )
    scorer = Faithfulness(llm=evaluator_llm)
    faithfulness_score = await scorer.single_turn_ascore(sample)
    return faithfulness_score

In [ ]:
import asyncio

# Initialize list to hold results
results = []

# Define an async function to run all evaluations
async def evaluate_all():
    tasks = []
    
    for index, row in qa_data.iterrows():
        context = [row.knowledge]
        question = row.question
        response = row.selected_answer
        
        
        # Store async tasks
        tasks.append(
            evaluate_hallucination(context, question, response)
        )

    # Run all evaluations concurrently
    faithfulness_scores = await asyncio.gather(*tasks)
    
    # Store results
    # for i in range(10):
    #     results.append({
    #         "question": qa_data.question[i],
    #         "context": [qa_data.knowledge[i]],
    #         "llm_answer": qa_data.hallucinated_answer[i],
    #         "faithfulness_score": faithfulness_scores[i]
    #     })
    
    results.append({
        "question": qa_data.question,
        "context": qa_data.knowledge,
        "response": qa_data.selected_answer,
        "faithfulness_score": faithfulness_scores,
        "hallucination_score": 1 - faithfulness_scores,
        'hallucinated_flag': qa_data.hallucinated_flag
    })

# Run the async function properly
asyncio.run(evaluate_all())

In [ ]:
# Convert to df, get hallucination score
results_df = pd.DataFrame(results)
results_df.head(10)